# Notebook A2: full-area stores for Notebook C

Builds what C needs beyond A's training sample: every 30 m pixel of every county, plus the coarse weather (history and SEAS5).

**Run order:** DEM correction → A → **A2** → B → C (D before C).

**Output** (in `F:\new_predict\VI_LSTM\county_store\`)

| File | Content |
|---|---|
| `<County>\<INDEX>.bin` | all months, `int16` |
| `<County>\qa_<INDEX>.bin` | QA, `uint8` (inputs with a QA band) |
| `<County>\coarse_hist.npz` | coarse historical weather |
| `<County>\coarse_forecast_<scenario>.npz` | raw SEAS5 (`mean`, `p10`, `p90`) |
| `manifest.json` | grid, scales, paths, months |

Same pixels, values and clipping as A. The forecast is raw SEAS5 (C and D correct it). Resumable.

In [ ]:
import os, re, glob, json, calendar, time, hashlib, shutil, warnings
import numpy as np, pandas as pd
import rasterio, rasterio.crs
from affine import Affine
from rasterio.enums import Resampling
from rasterio.warp import transform as warp_transform
from rasterio.windows import Window
from scipy import ndimage
from concurrent.futures import ThreadPoolExecutor

WORK_DIR   = r"F:\new_predict\VI_LSTM"                              # = WORK_DIR of Notebook A
STORE_DIR  = os.path.join(WORK_DIR, "county_store")                 # = STORE_DIR in Notebook C
MANIFEST   = os.path.join(STORE_DIR, "manifest.json")
STATIC_DIR = os.path.join(WORK_DIR, "_static")
CACHE_DIR  = os.path.join(WORK_DIR, "_cache"); os.makedirs(CACHE_DIR, exist_ok=True)
DATA_RAW     = os.path.join(WORK_DIR, "training_dataset.npz")
DATA_CLIPPED = os.path.join(WORK_DIR, "training_dataset_clipped.npz")

VI_ROOT   = r"F:\VI_prediction"
VI_SUBDIR = {"SAVI": os.path.join("SAVI", "tiff"), "MTVI2": os.path.join("MTVI2", "tiff")}
NODATA    = -9999.0

LANDSAT_FILE = os.path.join(VI_ROOT, "LANDSAT_STACK", "{county}", "LANDSAT_STACK_{county}_{year}_{month:02d}.tif")          # no QA band
S2_FILE  = os.path.join(VI_ROOT, "Sentinel_AllBands_2024", "{county}", "S2_{county}_{year}_{month:02d}.tif")                # band 3 = B4, 6 = B7, 7 = B8, 11 = fill_flag
CWV_DIR  = {"Kakamega": "C_WV_KACH", "Kilifi": "C_WV_KILIFI", "Nyeri": "C_WV_NYERI"}
CWV_FILE = os.path.join(VI_ROOT, "coastal_water_vapour", "{cwv_dir}", "S2_CWV_{county}_{year}_{month:02d}.tif")              # band 1 = B1, 2 = B9, 3 = fill_flag
EXTRA = {
    "L_COASTAL": (LANDSAT_FILE, 1, None), "L_BLUE": (LANDSAT_FILE, 2, None), "L_LST_K": (LANDSAT_FILE, 3, None),
    "S_RED": (S2_FILE, 3, 11), "S_NIR": (S2_FILE, 7, 11), "S_RE3": (S2_FILE, 6, 11),
    "S_COASTAL": (CWV_FILE, 1, 3), "S_WV": (CWV_FILE, 2, 3),
}
def band_of(idx):
    e = EXTRA.get(idx)
    return (e[1], e[2]) if e else (None, None)
HIST_ROOT = r"F:\VI_prediction\HISTORICAL WEATHER"
HIST_WX = {
    "rain": os.path.join(HIST_ROOT, "CHIRPS_DATA_PY_daily", "filled_daily", "{county}", "CHIRPS_{county}_{year}_{month:02d}_*.tif"),
    "temp_max": os.path.join(HIST_ROOT, "ERA5_LAND_TEMP_fixed", "{county}", "tmax", "{county}_tmax_{year}{month:02d}*.tif"),
    "temp_min": os.path.join(HIST_ROOT, "ERA5_LAND_TEMP_fixed", "{county}", "tmin", "{county}_tmin_{year}{month:02d}*.tif"),
    "rh":   os.path.join(HIST_ROOT, "DAILY_RH", "RH_{county_upper}", "daily_tifs", "{county}_RH_{year}-{month:02d}-*.tif"),
    "wind": os.path.join(HIST_ROOT, "WIND_DATA_PY_daily", "county_daily_direct", "{county}", "WIND_{county}_{year}_{month:02d}_*.tif"),
}
WIND_BAND = 1
MIN_DAY_FRAC = 0.8
LAPSE_T, LAPSE_TD = -6.5 / 1000, -2.0 / 1000
USE_RH_OFFSET = True
# clip rules of Notebook A, section 7b
HEAVY_TAILED = ["MCARI", "CVI", "PSRI", "L_COASTAL", "L_BLUE", "L_LST_K", "S_RED", "S_NIR", "S_RE3", "S_COASTAL", "S_WV"]; MARGIN = 1.0; PHYSICAL = {"CVI": (0.0, None)}; BOUNDED = (-1.0, 1.0)

IN_DIR = r"F:\VI_prediction\FORECASTED_WEATHER_BC_V2"
SCENARIOS = ["mean", "p10", "p90"]
FC_VARS = {"temp": ("temperature", "temp"), "rain": ("rainfall", "rain"), "rh": ("humidity", "rh"), "wind": ("wind", "wind_speed")}
N_FORECAST_MONTHS = 7

# ---- store layout and behaviour ----
TILE = 256
I16_NODATA = -32768
BAKE_RH_OFFSET = True
READ_WORKERS = 4
REBUILD_STORES = False
REBUILD_COARSE = False                       # True = re-read all daily weather files
N_CHECK = 3000
STORE_VERSION, WX_VERSION = "a2-v1", "a2-v1"

def vi_file(idx, c, y, m):
    if idx in EXTRA: return EXTRA[idx][0].format(county=c, year=y, month=m, cwv_dir=CWV_DIR.get(c, c))
    return os.path.join(VI_ROOT, VI_SUBDIR.get(idx, idx), c, f"{idx}_{c}_{y}_{m:02d}.tif")
static_path = lambda c: os.path.join(STATIC_DIR, f"{c}_static.tif")
dz_path     = lambda c: os.path.join(STATIC_DIR, f"{c}_dz.tif")
rhoff_path  = lambda c: os.path.join(STATIC_DIR, f"{c}_rhoff.json")
os.makedirs(STORE_DIR, exist_ok=True)

## 1. Inputs from Notebook A
Indices, months and counties are read from `training_dataset.npz`. The grid is A's static layer.

In [ ]:
for p in (DATA_RAW,):
    if not os.path.exists(p): raise FileNotFoundError(f"{p} not found: run Notebook A first")
Z0 = np.load(DATA_RAW, allow_pickle=False)
VIS          = [str(v) for v in Z0["vis"]]
MLAB         = [str(m) for m in Z0["months"]]
COUNTIES     = [str(c) for c in Z0["counties"]]
STATIC_NAMES = [str(s) for s in Z0["static_names"]]
WXN          = [str(w) for w in Z0["wx_names"]]
WX_UNITS     = [str(u) for u in Z0["wx_units"]]
VI_STATS     = Z0["vi_stats"]; STAT_NAMES = [str(s) for s in Z0["vi_stat_names"]]
if WXN != ["temp", "rain", "rh", "wind"]: raise RuntimeError(f"C's chain_at expects temp, rain, rh, wind; the dataset has {WXN}")
RAIN_TOTAL = WX_UNITS[1] == "mm/month"
MONTHS = [(int(m[:4]), int(m[5:7])) for m in MLAB]; M = len(MONTHS)
FC_LABELS = [str(pd.Period(MLAB[-1], "M") + i) for i in range(1, N_FORECAST_MONTHS + 1)]
FC_MONTHS = [(int(m[:4]), int(m[5:7])) for m in FC_LABELS]
print(f"{len(COUNTIES)} counties {COUNTIES} | {M} months {MLAB[0]}..{MLAB[-1]} | indices {VIS}")
print(f"forecast months {FC_LABELS} | rain unit: {WX_UNITS[1]} | static layers {STATIC_NAMES}")

G = {}
for c in COUNTIES:
    for p in (static_path(c), dz_path(c)):
        if not os.path.exists(p): raise FileNotFoundError(f"{p} not found: Notebook A (section 2) writes the static layers and dz")
    with rasterio.open(static_path(c)) as s:
        if s.crs is None or not s.crs.is_projected: raise RuntimeError(f"{c}: the static layer must be in a projected CRS")
        if s.count != len(STATIC_NAMES): raise RuntimeError(f"{c}: static layer has {s.count} bands, the dataset lists {STATIC_NAMES}")
        if list(s.descriptions) != STATIC_NAMES: print(f"  note {c}: band descriptions {list(s.descriptions)} differ from {STATIC_NAMES}; the order of the dataset is used")
        crs, tf, (h, w) = s.crs, s.transform, (s.height, s.width)
    ny, nx = -(-h // TILE), -(-w // TILE)
    rh_off = 0.0
    if USE_RH_OFFSET and BAKE_RH_OFFSET:
        if os.path.exists(rhoff_path(c)): rh_off = float(json.load(open(rhoff_path(c)))["rh_offset"])
        else: print(f"  note {c}: no RH offset file {rhoff_path(c)}; 0 used")
    G[c] = dict(crs=crs, transform=tf, shape=(h, w), ny=ny, nx=nx, nt=ny * nx, store_dir=os.path.join(STORE_DIR, c),
                static=static_path(c), dz=dz_path(c), rh_off=rh_off)
    os.makedirs(G[c]["store_dir"], exist_ok=True)
    print(f"{c}: grid {h} x {w} ({h * w / 1e6:.1f} M pixels) | {ny} x {nx} = {ny * nx} tiles of {TILE} | RH offset baked into the coarse RH: {rh_off:+.2f} %")

est = sum(M * g["nt"] * TILE * TILE * 2 * len(VIS) for g in G.values())
est_qa = sum(M * g["nt"] * TILE * TILE for g in G.values())
free = shutil.disk_usage(STORE_DIR).free
print(f"\nstores need about {est / 1e9:.0f} GB (+ {est_qa / 1e9:.1f} GB per index that has a QA band) | free on this drive: {free / 1e9:.0f} GB")
if est > 0.95 * free: print("!! probably not enough free disk space")

## 2. Helpers
Weather helpers come from A; `Coarse` and `chain_at` from C.

In [ ]:
A_MAGNUS, B_MAGNUS = 17.625, 243.04
def rh_from_td(t, td):
    return np.clip(100 * np.exp(A_MAGNUS * td / (B_MAGNUS + td)) / np.exp(A_MAGNUS * t / (B_MAGNUS + t)), 0, 100)
def td_from_rh(t, rh):
    g = np.log(np.clip(rh, 1, 100) / 100) + A_MAGNUS * t / (B_MAGNUS + t)
    return B_MAGNUS * g / (A_MAGNUS - g)

def fill_nearest(a):
    m = np.isnan(a)
    if not m.any() or m.all(): return a
    idx = ndimage.distance_transform_edt(m, return_distances=False, return_indices=True)
    return a[tuple(idx)]

def to_crs(xs, ys, src_crs, dst_crs):
    if rasterio.crs.CRS.from_user_input(src_crs) == rasterio.crs.CRS.from_user_input(dst_crs):
        return np.asarray(xs), np.asarray(ys)
    x2, y2 = warp_transform(src_crs, dst_crs, np.asarray(xs).tolist(), np.asarray(ys).tolist())
    return np.asarray(x2), np.asarray(y2)

class Pts:
    def __init__(self, xs, ys, crs):
        self.xs, self.ys, self.crs, self._c = np.asarray(xs), np.asarray(ys), crs, {}
    def in_crs(self, crs):
        k = str(crs)
        if k not in self._c: self._c[k] = to_crs(self.xs, self.ys, self.crs, crs)
        return self._c[k]

def sample_tif_nearest(path, pts, band=1):
    with rasterio.open(path) as s:
        x2, y2 = pts.in_crs(s.crs)
        cols, rows = (~s.transform) * (x2, y2)
        cols, rows = np.floor(cols).astype(int), np.floor(rows).astype(int)
        inside = (rows >= 0) & (rows < s.height) & (cols >= 0) & (cols < s.width)
        out = np.full(len(x2), np.nan, "float32")
        if not inside.any(): return out
        r0, r1, c0, c1 = rows[inside].min(), rows[inside].max() + 1, cols[inside].min(), cols[inside].max() + 1
        a = s.read(band, window=Window(c0, r0, c1 - c0, r1 - r0), masked=True).astype("float32").filled(np.nan)
        out[inside] = a[rows[inside] - r0, cols[inside] - c0]
    return out

def _date_of(path):
    for mt in re.finditer(r"(?<!\d)(\d{4})[-_]?(\d{2})[-_]?(\d{2})(?!\d)", os.path.basename(path)):
        y, mo, d = map(int, mt.groups())
        if 2000 <= y <= 2100 and 1 <= mo <= 12 and 1 <= d <= 31: return y, mo, d
    return None

def daily_files(var, county, y, m):
    tpl = HIST_WX[var]
    files = sorted(f for f in glob.glob(tpl.format(county=county, county_lower=county.lower(), county_upper=county.upper(), year=y, month=m)) if f.lower().endswith(".tif"))
    out = {}
    for f in files:
        d = _date_of(f)
        if d and d[0] == y and d[1] == m: out[d[2]] = f
    if not out and files: out = {i + 1: f for i, f in enumerate(files[:calendar.monthrange(y, m)[1]])}
    return out

def fix_units(var, a):
    if var.startswith("temp") and np.nanmean(a) > 100: a = a - 273.15          # Kelvin -> degC
    if var == "rh" and np.nanmax(a) <= 1.5: a = a * 100                         # fraction -> %
    if var == "rain": a = np.where(a < 0, np.nan, a)                            # CHIRPS no-data
    return a

def wind_band_of(src):
    if WIND_BAND is not None: return WIND_BAND
    if src.count == 1: return 1
    for i, d in enumerate(src.descriptions, 1):
        if d and "speed" in d.lower(): return i
    raise ValueError(f"cannot tell which band is wind speed in {src.name}: set WIND_BAND")

def _aggregate(paths, var, nd):
    acc = cnt = None
    for f in paths:
        with rasterio.open(f) as s:
            b = wind_band_of(s) if var == "wind" else 1
            a = s.read(b, masked=True).astype("float32").filled(np.nan); tf, crs = s.transform, s.crs
        a = fix_units(var, a); ok = np.isfinite(a)
        acc = np.where(ok, a, 0) if acc is None else acc + np.where(ok, a, 0)
        cnt = ok.astype("float32") if cnt is None else cnt + ok
    mean = np.where(cnt > 0, acc / np.maximum(cnt, 1), np.nan).astype("float32")
    arr = mean * nd if (var == "rain" and RAIN_TOTAL) else mean
    return fill_nearest(arr), tf, crs

def _monthly_single(key, county, y, m):
    var = "temp" if key.startswith("temp") else key
    files = daily_files(key, county, y, m); nd = calendar.monthrange(y, m)[1]
    if len(files) < MIN_DAY_FRAC * nd: return None
    return _aggregate(list(files.values()), var, nd)

def monthly_coarse(var, county, y, m):
    if var != "temp": return _monthly_single(var, county, y, m)
    hi, lo = _monthly_single("temp_max", county, y, m), _monthly_single("temp_min", county, y, m)
    if hi is None or lo is None: return None
    if hi[0].shape != lo[0].shape or hi[1] != lo[1]: raise ValueError(f"tmax and tmin grids differ for {county} {y}-{m:02d}")
    return ((hi[0] + lo[0]) / 2).astype("float32"), hi[1], hi[2]

def fc_files(var, county, scen, y, m):
    group, name = FC_VARS[var]; out = {}
    for f in glob.glob(os.path.join(IN_DIR, group, name, scen, f"{county}_{name}_{scen}_{y}-{m:02d}-*.tif")):
        mt = re.search(r"_(\d{4}-\d{2}-\d{2})\.tif$", f)                      # ignores *_stack.tif
        if mt: out[mt.group(1)] = f
    return out

def fc_monthly(var, county, scen, y, m):
    files = fc_files(var, county, scen, y, m); nd = calendar.monthrange(y, m)[1]
    if len(files) < MIN_DAY_FRAC * nd: return None
    return _aggregate([files[d] for d in sorted(files)], var, nd)

def save_coarse(path, d):
    out = {}
    for v in WXN:
        a, tf, crs = d[v]
        out[f"arr_{v}"] = np.asarray(a, "float32"); out[f"tf_{v}"] = np.array(tuple(tf)[:6], "float64"); out[f"crs_{v}"] = np.array(crs.to_wkt())
    tmp = path + ".part.npz"; np.savez_compressed(tmp, **out); os.replace(tmp, path)

def load_coarse(path):
    z = np.load(path); return {v: (z[f"arr_{v}"], Affine(*z[f"tf_{v}"]), rasterio.crs.CRS.from_wkt(str(z[f"crs_{v}"]))) for v in WXN}

class Coarse:
    def __init__(self, path):
        z = np.load(path); self.arr, self.tf, self.crs = {}, {}, {}
        for v in WXN:
            self.arr[v] = z[f"arr_{v}"]; self.tf[v] = Affine(*z[f"tf_{v}"]); self.crs[v] = rasterio.crs.CRS.from_wkt(str(z[f"crs_{v}"]))
        n = len(self.arr[WXN[0]])
        self.valid = np.array([all(np.isfinite(self.arr[v][i]).any() for v in WXN) for i in range(n)])
def _bilinear_index(shape, tf, xs, ys):
    cf, rf = (~tf) * (xs, ys)
    c, r = cf - 0.5, rf - 0.5
    h, w = shape
    c, r = np.clip(c, 0, w - 1), np.clip(r, 0, h - 1)
    c0, r0 = np.floor(c).astype(int), np.floor(r).astype(int)
    c1, r1 = np.minimum(c0 + 1, w - 1), np.minimum(r0 + 1, h - 1)
    return r0, r1, c0, c1, r - r0, c - c0
def chain_at(C, pts, dz, idx):
    idx = list(idx); out = np.full((len(pts.xs), len(idx), 4), np.nan, "float32")
    dz = np.nan_to_num(np.asarray(dz, "float32"))
    use = [j for j, i in enumerate(idx) if C.valid[i]]
    if not use: return out
    ii = np.array([idx[j] for j in use])[:, None]
    cache = pts.__dict__.setdefault("_bw", {})
    got = {}
    for v in WXN:
        key = (id(C), v)
        if key not in cache:
            x, y = pts.in_crs(C.crs[v]); cache[key] = _bilinear_index(C.arr[v].shape[1:], C.tf[v], x, y)
        r0, r1, c0, c1, fr, fc = cache[key]; A = C.arr[v]
        top = A[ii, r0[None, :], c0[None, :]] * (1 - fc) + A[ii, r0[None, :], c1[None, :]] * fc
        bot = A[ii, r1[None, :], c0[None, :]] * (1 - fc) + A[ii, r1[None, :], c1[None, :]] * fc
        got[v] = (top * (1 - fr) + bot * fr).astype("float32")
    Tm, RH = got["temp"], got["rh"]
    Td = td_from_rh(Tm, RH); Tc, Tdc = Tm + LAPSE_T * dz[None, :], Td + LAPSE_TD * dz[None, :]
    res = (Tc, got["rain"], rh_from_td(Tc, Tdc), got["wind"])
    for k in range(4): out[:, use, k] = res[k].T
    return out

## 3. VI stores
One `int16` store per county and index, clipped with A's rules.

In [ ]:
SKIP_BAD_FILES = True    # False = stop with the error
BAD_FILES = []
def pick_scale(lo, hi):
    m = max(abs(lo), abs(hi), 1e-9)
    for s in (30000, 20000, 10000, 5000, 2000, 1000, 500, 200, 100, 50, 20, 10, 5, 2, 1):
        if s * m <= 32000: return float(s)
    return 1.0

BOUNDS, SCALE, rows = {}, {}, []
for k, v in enumerate(VIS):                                   # the same rules as Notebook A, section 7b
    p01, p99, sd = (float(VI_STATS[k, STAT_NAMES.index(n)]) for n in ("p01", "p99", "std"))
    if v in HEAVY_TAILED and np.isfinite(p01) and np.isfinite(p99):
        span = p99 - p01; lo, hi = p01 - MARGIN * span, p99 + MARGIN * span
        plo, phi = PHYSICAL.get(v, (None, None))
        if plo is not None: lo = max(lo, plo)
        if phi is not None: hi = min(hi, phi)
    else:
        lo, hi = BOUNDED
    BOUNDS[v], SCALE[v] = (float(lo), float(hi)), pick_scale(lo, hi)
    rows.append(dict(index=v, lower=round(lo, 4), upper=round(hi, 4), scale=SCALE[v], step=1 / SCALE[v], step_over_std=(1 / SCALE[v]) / sd if sd > 0 else np.nan))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("\nstep_over_std = rounding step / standard deviation of the index values; it should be far below 0.01.")

def grid_offset(s, g):
    if s.crs is None or rasterio.crs.CRS.from_user_input(s.crs) != g["crs"]: return None
    t, gt = s.transform, g["transform"]
    if not np.allclose([t.a, t.b, t.d, t.e], [gt.a, gt.b, gt.d, gt.e], rtol=1e-9, atol=1e-9): return None
    dc, dr = (t.c - gt.c) / gt.a, (t.f - gt.f) / gt.e
    if abs(dc - round(dc)) > 1e-3 or abs(dr - round(dr)) > 1e-3: return None
    return int(round(dr)), int(round(dc))

import threading
_MAPS, _MAP_LOCK = {}, threading.Lock()
def grid_map(s, g):
    key = (id(g), str(s.crs), tuple(s.transform)[:6], s.height, s.width)
    with _MAP_LOCK:
        if key in _MAPS: return _MAPS[key]
        h, w = g["shape"]; inside = np.zeros((h, w), bool); R, C = [], []
        cols, inv = np.arange(w) + 0.5, ~s.transform
        for r0 in range(0, h, 512):
            r1 = min(r0 + 512, h); cc, rr = np.meshgrid(cols, np.arange(r0, r1) + 0.5)
            x, y = g["transform"] * (cc.ravel(), rr.ravel())
            x2, y2 = to_crs(x, y, g["crs"], s.crs)
            cf, rf = inv * (x2, y2)
            ci, ri = np.floor(cf).astype(np.int64), np.floor(rf).astype(np.int64)
            ok = (ri >= 0) & (ri < s.height) & (ci >= 0) & (ci < s.width)
            inside[r0:r1] = ok.reshape(r1 - r0, w); R.append(ri[ok].astype(np.int32)); C.append(ci[ok].astype(np.int32))
        _MAPS.clear(); _MAPS[key] = (inside, np.concatenate(R), np.concatenate(C))
        return _MAPS[key]

def read_on_grid(path, g, idx, want_qa):
    h, w = g["shape"]; Hp, Wp = g["ny"] * TILE, g["nx"] * TILE
    v = np.full((Hp, Wp), np.nan, "float32"); q = None
    band, qa_band = band_of(idx)
    with rasterio.open(path) as s:
        if idx in EXTRA: bands = [band, qa_band] if qa_band else [band]
        else: bands = [1, 2] if (want_qa and s.count >= 2) else [1]
        off = grid_offset(s, g)
        if off is not None:                                    # same grid: copy the overlap
            dr, dc = off
            r0, c0, r1, c1 = max(dr, 0), max(dc, 0), min(dr + s.height, h), min(dc + s.width, w)
            if len(bands) == 2: q = np.full((Hp, Wp), 255, "float32")
            if r1 > r0 and c1 > c0:
                d = s.read(bands, window=Window(c0 - dc, r0 - dr, c1 - c0, r1 - r0)).astype("float32")
                v[r0:r1, c0:c1] = d[0]
                if q is not None: q[r0:r1, c0:c1] = d[1]
        else:
            inside, rr, cc = grid_map(s, g)
            d = s.read(bands).astype("float32")
            vh = np.full((h, w), np.nan, "float32"); vh[inside] = d[0][rr, cc]; v[:h, :w] = vh
            if len(bands) == 2:
                qh = np.full((h, w), 255, "float32"); qh[inside] = d[1][rr, cc]; q = np.full((Hp, Wp), 255, "float32"); q[:h, :w] = qh
    if q is not None and idx.startswith("S_"):
        q = np.where(q == 1, 2, q)
    return v, q

def encode(v, q, lo, hi, sc, has_qa):
    ok = np.isfinite(v) & (v != NODATA)
    if has_qa:
        qq = q if q is not None else np.where(ok, 0, 255).astype("float32")
        ok &= np.isin(qq, (0, 2))
    n_clip = int(((v[ok] < lo) | (v[ok] > hi)).sum())
    val = np.rint(np.clip(np.where(ok, v, 0), lo, hi) * sc)
    i16 = np.where(ok, np.clip(val, -32767, 32767), I16_NODATA).astype("int16")
    return i16, (np.where(ok, qq, 255).astype("uint8") if has_qa else None), int(ok.sum()), n_clip

def to_tiles(a, g):
    return a.reshape(g["ny"], TILE, g["nx"], TILE).transpose(0, 2, 1, 3).reshape(g["nt"], TILE, TILE)

def files_signature(files):
    parts = []
    for f in files:
        if os.path.exists(f): st = os.stat(f); parts.append(f"{os.path.basename(f)}:{st.st_size}:{int(st.st_mtime)}")
    return hashlib.md5("|".join(parts).encode()).hexdigest()

def build_index_store(c, v):
    g = G[c]; lo, hi = BOUNDS[v]; sc = SCALE[v]
    files = [vi_file(v, c, y, m) for (y, m) in MONTHS]; have = [mi for mi, f in enumerate(files) if os.path.exists(f)]
    if not have: raise FileNotFoundError(f"{c} {v}: no monthly file found, e.g. {files[0]}")
    if v in EXTRA: has_qa = band_of(v)[1] is not None
    else:
        with rasterio.open(files[have[0]]) as s: has_qa = s.count >= 2
    marker = dict(version=STORE_VERSION, n_files=len(have), sig=files_signature(files), scale=sc, lo=lo, hi=hi, has_qa=bool(has_qa),
                  months=[MLAB[0], MLAB[-1]], tile=TILE, shape=list(g["shape"]))
    mpath = os.path.join(g["store_dir"], f"_done_{v}.json")
    pv, pq = os.path.join(g["store_dir"], f"{v}.bin"), os.path.join(g["store_dir"], f"qa_{v}.bin")
    if not REBUILD_STORES and os.path.exists(mpath) and os.path.exists(pv) and (not has_qa or os.path.exists(pq)):
        old = json.load(open(mpath))
        if {k: old.get(k) for k in marker if k != "n_used"} == marker:
            print(f"{c:9s} {v:6s} finished earlier ({len(have)}/{M} files): skipped"); return old
    if os.path.exists(mpath): os.remove(mpath)
    shp = (M, g["nt"], TILE, TILE); hv = set(have)
    mv = np.memmap(pv, np.int16, "w+", shape=shp)
    mq = np.memmap(pq, np.uint8, "w+", shape=shp) if has_qa else None
    def task(mi):
        ok_read = mi in hv
        if ok_read:
            try:
                vv, qq = read_on_grid(files[mi], g, v, has_qa)
                i16, q8, n_ok, n_clip = encode(vv, qq, lo, hi, sc, has_qa)
            except Exception as e:
                if not SKIP_BAD_FILES: raise
                BAD_FILES.append(files[mi]); ok_read = False
        if not ok_read:
            i16, q8, n_ok, n_clip = np.full((g["ny"] * TILE, g["nx"] * TILE), I16_NODATA, "int16"), (np.full((g["ny"] * TILE, g["nx"] * TILE), 255, "uint8") if has_qa else None), 0, 0
        mv[mi] = to_tiles(i16, g)
        if has_qa: mq[mi] = to_tiles(q8, g)
        return mi, n_ok, n_clip
    t0, n_ok_tot, n_clip_tot = time.time(), 0, 0
    with ThreadPoolExecutor(READ_WORKERS) as ex:
        for n, (mi, n_ok, n_clip) in enumerate(ex.map(task, range(M)), 1):
            n_ok_tot += n_ok; n_clip_tot += n_clip
            if n % 12 == 0 or n == M: print(f"    {c} {v}: {n}/{M} months ({time.time() - t0:.0f}s)", flush=True)
    mv.flush(); del mv
    if mq is not None: mq.flush(); del mq
    marker["n_used"] = n_ok_tot
    json.dump(marker, open(mpath, "w"))
    print(f"{c:9s} {v:6s} built: files {len(have)}/{M} | QA band {has_qa} | scale {sc:g} | usable values {n_ok_tot / 1e6:.0f} M, clipped {n_clip_tot} | {time.time() - t0:.0f}s" +
          ("" if len(have) == M else f"   <-- only {len(have)}/{M} monthly files found"))
    return marker

INDEX_INFO = {}
for c in COUNTIES:
    INDEX_INFO[c] = {v: build_index_store(c, v) for v in VIS}

if BAD_FILES:
    bad = sorted(set(BAD_FILES))
    open(os.path.join(WORK_DIR, "bad_vi_files_A2.txt"), "w").write("\n".join(bad))
    print(f"\nWARNING: {len(bad)} unreadable files were treated as missing:")
    print(*bad, sep="\n")

In [ ]:
import numpy as np, os

_REF = {}

def _ref_grid(c, var):
    key = (c, var)
    if key not in _REF:
        f = os.path.join(CACHE_DIR, f"coarse_hist_{c}_{WX_VERSION}_{MLAB[0]}_{MLAB[-1]}.npz")
        try:
            a, tr, _ = load_coarse(f)[var]
            _REF[key] = (tr, a.shape[-2], a.shape[-1])
        except Exception:
            _REF[key] = None
    return _REF[key]

if "_monthly_coarse_orig" not in globals():
    _monthly_coarse_orig = monthly_coarse

def monthly_coarse(var, c, y, m):
    out = _monthly_coarse_orig(var, c, y, m)
    ref = _ref_grid(c, var)
    if ref is None:
        return out
    rtr, ny, nx = ref
    arr, tr = out[0], out[1]
    if arr.shape[-2:] == (ny, nx) and tr == rtr:
        return out
    assert abs(tr.a - rtr.a) < 1e-6 and abs(tr.e - rtr.e) < 1e-6, ("cell size differs", var, y, m, tr, rtr)
    fx, fy = (tr.c - rtr.c) / rtr.a, (tr.f - rtr.f) / rtr.e
    ox, oy = int(round(fx)), int(round(fy))
    assert abs(fx - ox) < 0.05 and abs(fy - oy) < 0.05, ("grid not aligned", var, y, m, fx, fy)
    h, w = arr.shape[-2:]
    r0, r1 = max(0, oy), min(ny, oy + h)
    c0, c1 = max(0, ox), min(nx, ox + w)
    new = np.full(arr.shape[:-2] + (ny, nx), np.nan, dtype=np.float32)
    if r1 > r0 and c1 > c0:
        new[..., r0:r1, c0:c1] = arr[..., r0 - oy:r1 - oy, c0 - ox:c1 - ox]
    return (new, rtr) + tuple(out[2:]) 

## 4. Coarse weather: history
Monthly fields from the daily files, as in A. The RH offset is added.

In [ ]:
def regrid_nearest(a, tf, crs, tf0, shape0, crs0):
    h, w = shape0
    cc, rr = np.meshgrid(np.arange(w) + 0.5, np.arange(h) + 0.5)
    x, y = tf0 * (cc.ravel(), rr.ravel())
    x2, y2 = to_crs(x, y, crs0, crs)
    cf, rf = (~tf) * (x2, y2)
    ci, ri = np.floor(cf).astype(int), np.floor(rf).astype(int)
    ok = (ri >= 0) & (ri < a.shape[0]) & (ci >= 0) & (ci < a.shape[1])
    out = np.full(h * w, np.nan, "float32"); out[ok] = a[ri[ok], ci[ok]]
    return fill_nearest(out.reshape(h, w))

def assemble(results, n, have):
    out, empty, notes = {}, [], []
    for v in WXN:
        ref = have[v] if have is not None else None
        new = {i: r for (vv, i), r in results.items() if vv == v and r is not None}
        if ref is None:
            if not new: raise RuntimeError(f"no month at all with data for '{v}'")
            a0, tf0, crs0 = new[min(new)]; arr = np.full((n,) + a0.shape, np.nan, "float32")
        else:
            arr, tf0, crs0 = ref[0].copy(), ref[1], ref[2]
        moved = []
        for i, (a, tf, crs) in sorted(new.items()):
            same_crs = rasterio.crs.CRS.from_user_input(crs) == rasterio.crs.CRS.from_user_input(crs0)
            same = a.shape == arr.shape[1:] and np.allclose(tuple(tf)[:6], tuple(tf0)[:6], rtol=0, atol=1e-9) and same_crs
            if not same:
                if not same_crs or not np.allclose([tf.a, tf.e], [tf0.a, tf0.e], rtol=1e-6, atol=0):
                    raise RuntimeError(f"'{v}': month {MLAB[i]} has another CRS or pixel size than the other months ({tf.a:.5f} vs {tf0.a:.5f}); resample those files at the source")
                moved.append((i, a.shape, (tf.c - tf0.c) / tf0.a, (tf.f - tf0.f) / tf0.e))
                a = regrid_nearest(a, tf, crs, tf0, arr.shape[1:], crs0)
            arr[i] = a
        if moved:
            i0, sh, dx, dy = moved[0]
            notes.append(f"  NOTE '{v}': {len(moved)} month(s) ({MLAB[moved[0][0]]} .. {MLAB[moved[-1][0]]}) were on another grid and were regridded (nearest) onto the first month's grid; "
                         f"e.g. {MLAB[i0]}: shape {sh} vs {arr.shape[1:]}, origin shift ({dx:+.2f}, {dy:+.2f}) px")
        empty += [(v, i) for i in range(n) if not np.isfinite(arr[i]).any()]
        out[v] = (arr, tf0, crs0)
    for t in notes: print(t)
    return out, empty

def build_coarse_hist(c):
    cache = os.path.join(CACHE_DIR, f"coarse_hist_{c}_{WX_VERSION}_{MLAB[0]}_{MLAB[-1]}.npz")
    have = load_coarse(cache) if (os.path.exists(cache) and not REBUILD_COARSE) else None
    todo = [(v, i) for i in range(M) for v in WXN if have is None or not np.isfinite(have[v][0][i]).any()]
    print(f"{c}: {len(todo)} (variable, month) fields to read" + ("" if have is None else f" ({M * len(WXN) - len(todo)} cached)"), flush=True)
    res, t0 = {}, time.time()
    def task(t): v, i = t; return t, monthly_coarse(v, c, *MONTHS[i])
    with ThreadPoolExecutor(READ_WORKERS) as ex:
        for n, (t, r) in enumerate(ex.map(task, todo), 1):
            res[t] = r
            if n % 100 == 0: print(f"    {c}: {n}/{len(todo)} ({time.time() - t0:.0f}s)", flush=True)
    d, empty = assemble(res, M, have)
    if todo: save_coarse(cache, d)
    for v in WXN:
        miss = [MLAB[i] for vv, i in empty if vv == v]
        if miss: print(f"  WARNING {c}: no {v} for {len(miss)} months: {miss[:6]}{' ...' if len(miss) > 6 else ''}")
    last12 = sorted({MLAB[i] for v, i in empty if i >= M - 12})
    if last12: raise RuntimeError(f"{c}: months {last12} (inside the last 12) have no complete coarse weather; Notebook C cannot run until the daily files exist")
    return d

COARSE_HIST_PATH = {}
for c in COUNTIES:
    d = build_coarse_hist(c)
    if G[c]["rh_off"]: d["rh"] = (d["rh"][0] + np.float32(G[c]["rh_off"]), d["rh"][1], d["rh"][2])
    COARSE_HIST_PATH[c] = os.path.join(G[c]["store_dir"], "coarse_hist.npz"); save_coarse(COARSE_HIST_PATH[c], d)
    print(f"  {c}: wrote {COARSE_HIST_PATH[c]} | grids " + ", ".join(f"{v} {d[v][0].shape[1:]}" for v in WXN) + f" | RH offset added: {G[c]['rh_off']:+.2f}")

In [ ]:
for y, m in [(2025, 12), (2026, 1)]:
    f = list(daily_files("rh", "Kakamega", y, m).values())[0]
    with rasterio.open(f) as s: print(y, m, s.bounds)

## 5. Coarse weather: SEAS5
Same aggregation for `mean`, `p10`, `p90`. Raw, no correction.

In [ ]:
def build_coarse_forecast(c, scen):
    todo = [(v, j) for j in range(N_FORECAST_MONTHS) for v in WXN]
    def task(t): v, j = t; return t, fc_monthly(v, c, scen, *FC_MONTHS[j])
    res = {}
    with ThreadPoolExecutor(READ_WORKERS) as ex:
        for t, r in ex.map(task, todo): res[t] = r
    miss = [(v, FC_LABELS[j]) for (v, j), r in res.items() if r is None]
    if miss: return None, miss
    d, empty = assemble(res, N_FORECAST_MONTHS, None)
    if empty: return None, [(v, FC_LABELS[j]) for v, j in empty]
    if G[c]["rh_off"]: d["rh"] = (d["rh"][0] + np.float32(G[c]["rh_off"]), d["rh"][1], d["rh"][2])
    return d, []

FC_PATH = {c: {} for c in COUNTIES}
for c in COUNTIES:
    for scen in SCENARIOS:
        d, miss = build_coarse_forecast(c, scen)
        if d is None:
            print(f"{c:9s} {scen:5s} NOT built: missing or too few daily files for {miss[:6]}{' ...' if len(miss) > 6 else ''}"); continue
        p = os.path.join(G[c]["store_dir"], f"coarse_forecast_{scen}.npz"); save_coarse(p, d); FC_PATH[c][scen] = p
        n_days = {FC_LABELS[j]: len(fc_files("temp", c, scen, *FC_MONTHS[j])) for j in range(N_FORECAST_MONTHS)}
        print(f"{c:9s} {scen:5s} built | daily files per month (temp): {list(n_days.values())} | grids " + ", ".join(f"{v} {d[v][0].shape[1:]}" for v in WXN))
SCEN_OK = [s for s in SCENARIOS if all(s in FC_PATH[c] for c in COUNTIES)]
print("\nscenarios available for every county:", SCEN_OK)
if "mean" not in SCEN_OK: raise RuntimeError("the central scenario 'mean' could not be built for every county: check IN_DIR and the daily file names")

## 6. Manifest
Writes `manifest.json` for Notebook C.

In [ ]:
MAN = dict(version=STORE_VERSION, created=time.strftime("%Y-%m-%d %H:%M"), tile=TILE, i16_nodata=I16_NODATA, months=MLAB, forecast_months=FC_LABELS,
           scenarios=SCEN_OK, static_names=STATIC_NAMES, wx_names=WXN, wx_units=WX_UNITS, lapse=dict(temp=LAPSE_T, dewpoint=LAPSE_TD),
           clip=dict(heavy_tailed=HEAVY_TAILED, margin=MARGIN, physical={k: list(v) for k, v in PHYSICAL.items()}, bounded=list(BOUNDED)), counties={})
for c in COUNTIES:
    g = G[c]
    MAN["counties"][c] = dict(
        crs=g["crs"].to_wkt(), transform=[float(x) for x in tuple(g["transform"])[:6]], shape=[int(g["shape"][0]), int(g["shape"][1])],
        ny=int(g["ny"]), nx=int(g["nx"]), n_tiles=int(g["nt"]), static=g["static"], dz=g["dz"], store_dir=g["store_dir"], rh_offset_baked=float(g["rh_off"]),
        indices={v: dict(has_qa=bool(INDEX_INFO[c][v]["has_qa"]), scale=float(INDEX_INFO[c][v]["scale"]), lo=INDEX_INFO[c][v]["lo"], hi=INDEX_INFO[c][v]["hi"],
                         n_files=int(INDEX_INFO[c][v]["n_files"])) for v in VIS},
        coarse_hist=COARSE_HIST_PATH[c], coarse_forecast={s: FC_PATH[c][s] for s in SCEN_OK})
tmp = MANIFEST + ".part"; json.dump(MAN, open(tmp, "w"), indent=1); os.replace(tmp, MANIFEST)
print("wrote", MANIFEST)
print(pd.DataFrame({c: {v: ("QA, " if MAN["counties"][c]["indices"][v]["has_qa"] else "") + f"scale {MAN['counties'][c]['indices'][v]['scale']:g}, {MAN['counties'][c]['indices'][v]['n_files']}/{M} files" for v in VIS} for c in COUNTIES}).to_string())

## 7. Checks
- **(a)** Stores vs the training sample.
- **(b)** Coarse history vs the training weather.
- **(c)** SEAS5 vs history (catches unit errors).

In [ ]:
if not os.path.exists(DATA_CLIPPED):
    print("training_dataset_clipped.npz not found (Notebook A, section 7b): checks (a) and (b) are skipped")
    CHK = None
else:
    Zc = np.load(DATA_CLIPPED, allow_pickle=False)
    if [str(v) for v in Zc["vis"]] != VIS or [str(m) for m in Zc["months"]] != MLAB: raise RuntimeError("the clipped dataset and the raw dataset have different indices or months")
    cid, rw, cl = Zc["county_id"].astype(int), Zc["row"].astype(int), Zc["col"].astype(int)
    rng = np.random.default_rng(1); CHK = {}
    for ci, c in enumerate(COUNTIES):
        idx = np.flatnonzero(cid == ci); CHK[c] = np.sort(rng.choice(idx, min(N_CHECK, len(idx)), replace=False))
    sel_all = np.concatenate([CHK[c] for c in COUNTIES])
    VI_S, QA_S, WX_S, DZ_S, X_S, Y_S = (Zc[k][sel_all] for k in ("vi", "qa", "wx", "dz", "x", "y"))
    pos = {c: np.arange(len(sel_all))[np.isin(sel_all, CHK[c])] for c in COUNTIES}

    # ---- (a) stores vs the training sample ----
    out = []
    for c in COUNTIES:
        g = G[c]; p = pos[c]; r, cc = rw[sel_all[p]], cl[sel_all[p]]
        t, rl, cl_ = (r // TILE) * g["nx"] + cc // TILE, r % TILE, cc % TILE
        for k, v in enumerate(VIS):
            info = MAN["counties"][c]["indices"][v]; shp = (M, g["nt"], TILE, TILE)
            raw = np.asarray(np.memmap(os.path.join(g["store_dir"], f"{v}.bin"), np.int16, "r", shape=shp)[:, t, rl, cl_])          # [M, n]
            val = np.where(raw == I16_NODATA, np.nan, raw / info["scale"]).T                                                          # [n, M]
            s = VI_S[p][:, :, k]; both = np.isfinite(val) & np.isfinite(s)
            qa_bad = np.nan
            if info["has_qa"]:
                qs = np.asarray(np.memmap(os.path.join(g["store_dir"], f"qa_{v}.bin"), np.uint8, "r", shape=shp)[:, t, rl, cl_]).T
                qa_bad = float((qs != QA_S[p][:, :, k]).mean())
            out.append(dict(county=c, index=v, n=int(np.isfinite(s).sum()), nan_pattern_mismatch=float((np.isfinite(val) != np.isfinite(s)).mean()),
                            max_abs_diff=float(np.abs(val - s)[both].max()) if both.any() else np.nan, allowed=0.5 / info["scale"] * 1.01 + 1e-6, qa_mismatch=qa_bad))
    TAB_A = pd.DataFrame(out); print("(a) stores vs training sample at the sample pixels:"); print(TAB_A.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
    bad = TAB_A[(TAB_A.nan_pattern_mismatch > 1e-3) | (TAB_A.max_abs_diff > TAB_A.allowed) | (TAB_A.qa_mismatch.fillna(0) > 1e-3)]
    print("\n(a) OK: the stores reproduce the training sample." if len(bad) == 0 else "\n(a) !! these rows disagree with the training sample (check VI_ROOT / VI_SUBDIR, the clip rules, or whether the files changed since Notebook A):\n" + bad.to_string(index=False))

    out = []
    for c in COUNTIES:
        p = pos[c]; Ch = Coarse(COARSE_HIST_PATH[c]); pts = Pts(X_S[p], Y_S[p], G[c]["crs"])
        f = chain_at(Ch, pts, DZ_S[p], list(range(M)))
        for j, w in enumerate(WXN):
            a, b = f[:, :, j], WX_S[p][:, :, j]; both = np.isfinite(a) & np.isfinite(b); d = np.abs(a - b)[both]
            out.append(dict(county=c, var=w, n=int(both.sum()), nan_mismatch=float((np.isfinite(a) != np.isfinite(b)).mean()), max_abs_diff=float(d.max()) if d.size else np.nan,
                            p99_abs_diff=float(np.percentile(d, 99)) if d.size else np.nan, mean_diff=float((a - b)[both].mean()) if d.size else np.nan))
    TAB_B = pd.DataFrame(out); print("\n(b) coarse history through chain_at vs the training weather (temp degC, rain mm/month, rh %, wind m/s):"); print(TAB_B.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    tol = {"temp": 0.02, "rain": 0.1 if RAIN_TOTAL else 0.01, "rh": 0.5, "wind": 0.02}
    bad = TAB_B[[r["max_abs_diff"] > tol[r["var"]] or r["nan_mismatch"] > 1e-3 for _, r in TAB_B.iterrows()]]
    print("\n(b) OK: the history weather matches the training weather." if len(bad) == 0 else "\n(b) !! larger differences than expected (HIST_WX patterns, RAIN unit, USE_RH_OFFSET or dz differ from Notebook A):\n" + bad.to_string(index=False))

In [ ]:
if CHK is None:
    print("(c) skipped: needs the sample pixels of the clipped dataset")
else:
    cal = np.array([m for _, m in MONTHS]); rows = []; means = {}
    for c in COUNTIES:
        p = pos[c]; pts = Pts(X_S[p], Y_S[p], G[c]["crs"])
        for s in SCEN_OK:
            f = chain_at(Coarse(FC_PATH[c][s]), pts, DZ_S[p], list(range(N_FORECAST_MONTHS)))
            for j, w in enumerate(WXN):
                means[(c, s, w)] = np.nanmean(f[:, :, j], axis=0)
                if s != "mean": continue
                for h in range(N_FORECAST_MONTHS):
                    cols = np.flatnonzero(cal == FC_MONTHS[h][1] )
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore", RuntimeWarning); py = np.nanmean(WX_S[p][:, cols, j], axis=0)
                    py = py[np.isfinite(py)]
                    if len(py) < 1: continue
                    mu, sd = py.mean(), (py.std(ddof=1) if len(py) > 1 else np.nan); fc = float(means[(c, s, w)][h])
                    rows.append(dict(county=c, var=w, month=FC_LABELS[h], forecast=fc, hist_mean=mu, hist_min=py.min(), hist_max=py.max(), z=(fc - mu) / max(sd, 1e-6), ratio=fc / mu if abs(mu) > 1e-6 else np.nan))
    CMP = pd.DataFrame(rows, columns=["county", "var", "month", "forecast", "hist_mean", "hist_min", "hist_max", "z", "ratio"]); print("(c) raw 'mean' scenario at the sample pixels vs history of the same calendar month:"); print(CMP.round(2).to_string(index=False))
    bad = CMP[(CMP.z.abs() > 3) | ((CMP["var"] == "rain") & ((CMP.ratio < 0.2) | (CMP.ratio > 5)))]
    print("\n(c) rows with |z| > 3 or a rain ratio outside 0.2-5 (a few degrees of SEAS5 warm bias are expected and are removed by Notebook D; magnitudes like 30x are unit errors):")
    print(bad.round(2).to_string(index=False) if len(bad) else "  none")
    r = CMP[CMP["var"] == "rain"]
    if len(r): print(f"  median forecast / history ratio for rain: {r.ratio.median():.2f} (about 1 expected)")
    if {"p10", "p90"} <= set(SCEN_OK):
        wrong = [(c, w, FC_LABELS[h]) for c in COUNTIES for w in WXN for h in range(N_FORECAST_MONTHS)
                 if not (means[(c, "p10", w)][h] <= means[(c, "mean", w)][h] + 1e-6 <= means[(c, "p90", w)][h] + 2e-6)]
        print(f"\n  p10 <= mean <= p90 holds for all counties, variables and months." if not wrong else f"\n  !! p10 <= mean <= p90 fails for {len(wrong)} county/variable/month cells, e.g. {wrong[:4]}: check the scenario folders")

## Notes
- Keep the folder where it is, or re-run section 6 after moving it.
- New VI files: re-run sections 3 and 6.
- New daily weather: re-run section 4.
- New SEAS5 forecast: re-run A and B, then sections 5 and 6.
- RH offset changed: set `REBUILD_COARSE = True` and re-run sections 4 to 6.